In [2]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np

LOG.propagate = False

In [ ]:
ble = get_ble_controller()
ble.connect()

In [ ]:
# Task1
ble.send_command(CMD.ECHO,"HiHello")
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

In [4]:
# Task2
ble.send_command(CMD.SEND_THREE_FLOATS,"0.11|0.22|0.33")

In [5]:
# Task3
ble.send_command(CMD.GET_TIME_MILLIS, "")
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

T:43172


In [78]:
# Task4
def notif_handler(uuid,byte_array):
    s = ble.bytearray_to_string(byte_array)

    if s.startswith("T:"):
        t_ms = int(s[2:])
        print(f"Timestamp = {t_ms}")

ble.start_notify(ble.uuid["RX_STRING"], notif_handler)

In [79]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [ ]:
# Task5
def notif_handler(uuid,byte_array):
    s = ble.bytearray_to_string(byte_array)

    if s.startswith("T:"):
        t_ms = int(s[2:])
        print(f"Timestamp = {t_ms}")

ble.start_notify(ble.uuid["RX_STRING"], notif_handler)
ble.send_command(CMD.TIME_LOOP,"")

In [ ]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [82]:
# Task6
timestamps = []

def data_handler(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)      

    if s.startswith("T:"):
        timestamps.append(int(s[2:]))

ble.start_notify(ble.uuid["RX_STRING"], data_handler)
ble.send_command(CMD.SEND_TIME_DATA, "")

In [ ]:
print(len(timestamps))
# print(timestamps)

In [57]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [ ]:
# Task7
timestamps = []
temps = []

def timetemp_handler(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)
    if s.startswith("Time(ms):"):
        part_time, part_temp = s.split(",")   
        time = part_time.split(":")[1]
        temp = part_temp.split(":")[1]
        timestamps.append(int(time))
        temps.append(float(temp))
        print(s)

ble.start_notify(ble.uuid["RX_STRING"], timetemp_handler)
ble.send_command(CMD.GET_TEMP_READINGS, "")

In [ ]:
print(len(timestamps))
print(len(temps))

In [89]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [43]:
# task1
recv_timestamps = []
recv_messages = []

def reply5_handler(uuid, byte_array):
    msg = ble.bytearray_to_string(byte_array)
    recv_timestamps.append(time.perf_counter())
    recv_messages.append(msg)

ble.start_notify(ble.uuid["RX_STRING"], reply5_handler)
t_send = time.perf_counter()
ble.send_command(CMD.REPLY_NB, "120")

In [ ]:
if recv_timestamps:
    rtt_ms = (recv_timestamps[0] - t_send) * 1000
    reply_len = len(recv_messages[0])

    print("reply =", recv_messages[0])
    print("reply_length (bytes) =", reply_len)
    print("RTT_ms ≈", rtt_ms)

    # Data rate estimation (bytes per second)
    rtt_s = rtt_ms / 1000
    data_rate_Bps = reply_len / rtt_s

    print("Estimated data rate ≈", data_rate_Bps, "bytes/s")

In [45]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [49]:
# task2
received = []

def handler(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)
    if s.isdigit():
        received.append(int(s))

ble.start_notify(ble.uuid["RX_STRING"], handler)
ble.send_command(CMD.RELIABILITY, "")

In [ ]:
print("Received:", len(received))
print("Missing:", sorted(set(range(1000)) - set(received)))

In [49]:
ble.stop_notify(ble.uuid["RX_STRING"])

In [7]:
# 2) 存储容器
tof_front_t, tof_front_d = [], []
tof_right_t, tof_right_d = [], []
imu_rows = []   # 每行先存成 list，最后再拆列

current_section = None
dump_done = False
start_seen = False

def parse_line(line: str):
    global current_section, dump_done, start_seen

    line = line.strip()
    if not line:
        return

    # ---- markers ----
    if line.startswith("SENSORS_LOG_STARTED"):
        start_seen = True
        return

    if line.startswith("LOG_BEGIN"):
        current_section = None
        return

    if line.startswith("TOF_FRONT"):
        current_section = "TOF_FRONT"
        return

    if line.startswith("TOF_RIGHT"):
        current_section = "TOF_RIGHT"
        return

    if line.startswith("IMU"):
        # 这一行既可能是 "IMU,header..." 也可能是数据行（极少见）
        # 你 Arduino 里发的是 header：IMU,t_ms,...
        if "t_ms" in line:
            current_section = "IMU"
            return

    if line.startswith("LOG_DONE"):
        dump_done = True
        return

    # ---- data lines (csv) ----
    try:
        parts = line.split(",")

        if current_section == "TOF_FRONT":
            # t_ms, dist_mm
            t = int(parts[0])
            d = int(parts[1])
            tof_front_t.append(t)
            tof_front_d.append(d)
            return

        if current_section == "TOF_RIGHT":
            t = int(parts[0])
            d = int(parts[1])
            tof_right_t.append(t)
            tof_right_d.append(d)
            return

        if current_section == "IMU":
            # t_ms,pitch_acc,roll_acc,pitch_g,roll_g,yaw_g,pitch_cf,roll_cf
            # 共 8 列
            if len(parts) >= 8:
                row = [float(x) for x in parts[:8]]
                imu_rows.append(row)
            return

    except Exception:
        # 如果偶尔收到奇怪行/残缺行，直接跳过不让程序崩
        return

def notif_handler(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)
    # 有时一次 notify 里可能带多行（取决于实现），稳妥起见 splitlines
    for line in s.splitlines():
        parse_line(line)

ble.start_notify(ble.uuid["RX_STRING"], notif_handler)

# 4) 发命令：开始采集（板子采 5s）
dump_done = False
start_seen = False
tof_front_t.clear(); tof_front_d.clear()
tof_right_t.clear(); tof_right_d.clear()
imu_rows.clear()

ble.send_command(CMD.START_SENSORS_LOG, "")

# 等待采集结束：你的板子 LOG_MS=5000ms，给一点余量
time.sleep(6.0)

# 5) 发命令：dump 数据
ble.send_command(CMD.DUMP_SENSORS_LOG, "")

# 6) 等待收到 LOG_DONE（最多等 20s，防止卡死）
t_wait0 = time.time()
while not dump_done and (time.time() - t_wait0) < 20:
    time.sleep(0.05)

ble.stop_notify(ble.uuid[notify_uuid_key])

print("TOF_FRONT samples:", len(tof_front_t))
print("TOF_RIGHT samples:", len(tof_right_t))
print("IMU samples:", len(imu_rows))

# 7) 把 imu_rows 拆成列（方便后面画图）
if imu_rows:
    imu_t       = [r[0] for r in imu_rows]
    pitch_acc   = [r[1] for r in imu_rows]
    roll_acc    = [r[2] for r in imu_rows]
    pitch_g     = [r[3] for r in imu_rows]
    roll_g      = [r[4] for r in imu_rows]
    yaw_g       = [r[5] for r in imu_rows]
    pitch_cf    = [r[6] for r in imu_rows]
    roll_cf     = [r[7] for r in imu_rows]

Exception: Not connected to a BLE device

In [ ]:
ble.disconnect()